# AEI last version
$(\omega - m\Omega_\theta)^2 = \Omega_r^2 + \frac{2B_0 |k|}{\Sigma r} + \frac{k^2 c_s^2}{r^2}$

- $B_0$ campo imperturbato, prpendicolare al disco. legato al materiale del disco - assumiamo legge di potenza con esponente 5/4 come da Shakura&Sunyaev
- $\Sigma$ densità superficiale, legge di potenza con esponente ..
- assumo disco sottile, con aspect ratio $H/r = HOR$
- $c_s$ velocità del suono - in disco sottile è dato approssimativamente dalla formula sotto
- k = numero d'onda. lo prendiamo come parametro liberto ma poi controlliamo che stia in range ragionevole ($k \approx 1/H$ con H scala verticale tipica del disco)
- m: generalmente il modo eccitato ha un solo braccio, perciò poniamo m=1 per ridurre il numro di parametri


> $B_0 = B_{00} (r/r_{H})^{-\alpha_B}$. $B_{00}$ valore di riferimento sull'orizzonte degli eventi ($r_H$) del BH

> $\Sigma = \Sigma_0 (r/r_{in})^{-\alpha_\Sigma}$. $\Sigma_0$ riferimento al borndo interno del disco

> $c_s \approx (H/r) v_\varphi = (H/r) r \Omega_\varphi$

> $0.1/r < |k| < 10/r \ ,\ $ 
> $\beta = \frac{ 8 \pi p}{B^2_0} \approx \frac{ 8 \pi \Sigma c_s^2}{B^2_0} \leq 1 \ ,\ $
> $\frac{d}{dr}\left( \frac{\Omega\Sigma}{B^2_0} \right) > 0$

## settings

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from aei_2.aei_setup import (
    solve_k_aei, compute_beta, compute_dQdr,
    r_ilr, r_olr, r_corotation, _make_interp,
    make_disk, get_resonances
)

from setup import M_BH, NU0, r_isco, Rg_SUN, set_style, fix_spines
from aei_2.simple_disc import disk_model_simple

In [ ]:
from matplotlib.colors import LogNorm
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import LogLocator, LogFormatterSciNotation
import matplotlib.cm as cm
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
import matplotlib.gridspec as gridspec

set_style()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# GLOBAL SETTINGS
# ═══════════════════════════════════════════════════════════════════════════════
A_GRID      = np.linspace(-1, 1, 100)   # spin grid
N_R        = 200   # radial points for full profile (ISCO → OLR)
N_R_CAV    = 100   # radial points for cavity (ISCO → ILR)

# Reference values for dQ/dr plots (only sign matters, not amplitude)
B00_REF    = 1e4
SIGMA0_REF = 1e4

## simple

In [ ]:
# SIMPLE ====================================================================

ALP_B = 5 / 4    # Simple-v1 magnetic field exponent
ALP_S = 3 / 5    # Simple-v1 surface density exponent

CONFIGS = [
    dict(mm=1, hor=0.05,  color='#3b82f6', ls='-',  label=r'$m=1,\ H/r=0.05$'),
    dict(mm=1, hor=0.001, color='#f97316', ls='-',  label=r'$m=1,\ H/r=10^{-3}$'),
    dict(mm=2, hor=0.05,  color='#22c55e', ls='--', label=r'$m=2,\ H/r=0.05$'),
    dict(mm=2, hor=0.001, color='#ef4444', ls='--', label=r'$m=2,\ H/r=10^{-3}$'),
]

B00_GRID    = np.logspace(1,  8, 48)          # B00 [G]
SIGMA0_GRID = np.logspace(2,  7, 36)          # Sigma0 [g/cm²]

In [ ]:
def dQdr_profile(r_vec, a, B00, Sigma0, hor):
    """
    Compute dQ/dr = d/dr [Omega_phi * Sigma / B0²].
    The sign is independent of B00, Sigma0 (positive multiplicative constant).
    H/r (hor) does not enter Q at all.
    """
    res = make_disk(disk_model_simple, (r_vec, a, B00, Sigma0, ALP_B, ALP_S, hor))
    B0i = _make_interp(r_vec, res[0])
    Si  = _make_interp(r_vec, res[1])
    return compute_dQdr(r_vec, a, B0i, Si, M_BH)

# ═══════════════════════════════════════════════════════════════════════════════
# 2 ── MAIN SCAN
# ═══════════════════════════════════════════════════════════════════════════════

def cavity_scan(mm, hor, verbose=False):
    """
    Three-step filter over the full (a, B00, Sigma0) grid.

    Step A  dQ/dr > 0 for ALL r ∈ [CR-eps, CR+eps]
            Computed ONCE per (a, mm) — independent of B00, Sigma0, hor.
            The B00/Sigma0 loop is entirely skipped for failing spins.

    Step B  beta(r) ≤ 1 for ALL r ∈ [ISCO, OLR]
            Depends on Sigma0/B00² and hor.

    Step C  1 ≤ k(r) ≤ 1/hr(r) for ALL r ∈ [ISCO, ILR]
            Solves the AEI dispersion relation point by point.

    Returns
    -------
    pd.DataFrame with one row per (a, B00, Sigma0).
    Columns: a, B00, Sigma0, r_ILR, r_OLR, r_CR,
             dQdr_CR, pass_shear, pass_beta, pass_wkb,
             beta_max, k_med, k_min_val, k_max_val, frac_wkb_ok
    """
    records = []
    n_spin = len(A_GRID)
    n_par  = len(B00_GRID) * len(SIGMA0_GRID)

    for i_a, a_val in enumerate(A_GRID):
        rISCO = float(r_isco(a_val))
        r_ILR, r_OLR, r_CR = get_resonances(a_val, mm)

        if verbose:
            print(f'  spin {i_a+1}/{n_spin}  a={a_val:.2f} '
                  f'| r_ILR={r_ILR:.1f}  r_OLR={r_OLR:.1f}  r_CR={r_CR:.1f}',
                  end='  ')

        # ── sanity check on resonance radii ───────────────────────────────
        if not (np.isfinite(r_OLR) and np.isfinite(r_ILR)):
            if verbose:
                print('SKIP (resonances not found)')
            # fill records with failed rows
            for B00 in B00_GRID:
                for S0 in SIGMA0_GRID:
                    records.append(dict(a=a_val, B00=B00, Sigma0=S0,
                                        r_ILR=np.nan, r_OLR=np.nan, r_CR=r_CR,
                                        dQdr_CR=np.nan,
                                        pass_shear=False, pass_beta=False,
                                        pass_wkb=False))
            continue

        if r_ILR <= rISCO * 1.001:
            if verbose:
                print('SKIP (ILR inside ISCO)')
            for B00 in B00_GRID:
                for S0 in SIGMA0_GRID:
                    records.append(dict(a=a_val, B00=B00, Sigma0=S0,
                                        r_ILR=r_ILR, r_OLR=r_OLR, r_CR=r_CR,
                                        dQdr_CR=np.nan,
                                        pass_shear=False, pass_beta=False,
                                        pass_wkb=False))
            continue

        # ── Step A: dQ/dr (independent of B00, Sigma0, hor) ───────────────
        r_olr_cap = min(r_OLR, rISCO * 500.0)
        r_full    = np.geomspace(rISCO*1.001, r_olr_cap, N_R)

        r_cr_eps = np.linspace(r_CR*0.95, r_CR*1.05, 20)
        dq_full  = dQdr_profile(r_cr_eps, a_val, B00_REF, SIGMA0_REF, hor)
        dqdr_at_CR = float(np.interp(r_CR, r_cr_eps, dq_full))
        pass_shear = bool(dqdr_at_CR > 0)

        if verbose:
            shear_str = 'SHEAR OK' if pass_shear else 'shear FAIL'
            print(shear_str)

        if not pass_shear:
            for B00 in B00_GRID:
                for S0 in SIGMA0_GRID:
                    records.append(dict(a=a_val, B00=B00, Sigma0=S0,
                                        r_ILR=r_ILR, r_OLR=r_OLR, r_CR=r_CR,
                                        dQdr_CR=dqdr_at_CR,
                                        pass_shear=False, pass_beta=False,
                                        pass_wkb=False))
            continue

        # ── radial grids for beta & cavity ─────────────────────────────────
        r_ilr_cap = min(r_ILR*0.98, r_olr_cap)
        r_cavity  = np.geomspace(rISCO*1.01, r_ilr_cap, N_R_CAV)

        # ── Steps B & C: loop over (B00, Sigma0) ──────────────────────────
        for B00 in B00_GRID:
            for S0 in SIGMA0_GRID:

                # ── Step B: beta ≤ 1 in [ISCO, OLR] ─────────────────────
                res_full = make_disk(disk_model_simple, (r_full, a_val, B00, S0, ALP_B, ALP_S, hor))
                hr_full  = res_full[3]   # per-point H/r from model
                beta_full = compute_beta(res_full[0], res_full[1], res_full[2],
                                         r_full, hr_full, M_BH)
                pass_beta = bool(np.all(beta_full <= 1.0))
                beta_max  = float(np.max(beta_full))

                if not pass_beta:
                    records.append(dict(a=a_val, B00=B00, Sigma0=S0,
                                        r_ILR=r_ILR, r_OLR=r_OLR, r_CR=r_CR,
                                        dQdr_CR=dqdr_at_CR,
                                        pass_shear=True, pass_beta=False,
                                        pass_wkb=False, beta_max=beta_max))
                    continue

                # ── Step C: WKB in cavity [ISCO, ILR] ────────────────────
                res_cav  = make_disk(disk_model_simple, (r_cavity, a_val, B00, S0, ALP_B, ALP_S, hor))
                B0_c, Sg_c, cs_c, hr_c = res_cav[:4]

                k_arr = solve_k_aei(r_cavity, a_val, B0_c, Sg_c, cs_c,
                                    m=mm, M=M_BH)
                k_max_arr = 1 / np.maximum(hr_c, 1e-10)  # 1/hr(r)

                wkb_ok   = (np.isfinite(k_arr)
                            & (k_arr >= mm)
                            & (k_arr <= k_max_arr))
                pass_wkb = bool(wkb_ok.mean() >= 0.95)
                frac_ok  = float(wkb_ok.sum() / len(wkb_ok))

                k_valid = k_arr[np.isfinite(k_arr)]
                records.append(dict(
                    a=a_val, B00=B00, Sigma0=S0,
                    r_ILR=r_ILR, r_OLR=r_OLR, r_CR=r_CR,
                    dQdr_CR=dqdr_at_CR,
                    pass_shear=True, pass_beta=True, pass_wkb=pass_wkb,
                    beta_max=beta_max,
                    frac_wkb_ok=frac_ok,
                    k_med=float(np.nanmedian(k_arr)),
                    k_min_val=float(np.nanmin(k_arr)) if k_valid.size else np.nan,
                    k_max_val=float(np.nanmax(k_arr)) if k_valid.size else np.nan,
                ))

    return pd.DataFrame(records)


# ═══════════════════════════════════════════════════════════════════════════════
# 3 ── RUN ALL CONFIGS
# ═══════════════════════════════════════════════════════════════════════════════

def run_all():
    """Run cavity_scan for all 4 (m, H/r) configurations."""
    all_results = {}
    for cfg in CONFIGS:
        mm, hor = cfg['mm'], cfg['hor']
        print(f"\n{'='*65}")
        print(f"  Config: m={mm}, H/r={hor}")
        print(f"{'='*65}")
        df = cavity_scan(mm, hor, verbose=False)
        all_results[(mm, hor)] = df

        n      = len(df)
        n_sh   = df['pass_shear'].sum()
        n_be   = df.get('pass_beta',  pd.Series(dtype=bool)).sum() if 'pass_beta' in df else 0
        n_wkb  = df.get('pass_wkb',   pd.Series(dtype=bool)).sum() if 'pass_wkb'  in df else 0
        print(f'\n  Total combos : {n}')
        print(f'  Pass shear   : {n_sh:>6}  ({100*n_sh/n:.1f}%)')
        print(f'  + beta ≤ 1   : {n_be:>6}  ({100*n_be/n:.1f}%)')
        print(f'  + WKB cavity : {n_wkb:>6}  ({100*n_wkb/n:.1f}%)')

    return all_results

#### calc

In [ ]:
# ── step 1: full scan ─────────────────────────────────────────────────────
print('\n> Running cavity scan for all configs...')
all_results = run_all()

#### plot 1 - 3 successivi

In [ ]:
cfg = CONFIGS[1]
mm, hor = cfg['mm'], cfg['hor']
df = all_results[(mm, hor)]

stages = [
    ('pass_shear', r'$dQ/dr > 0$ at $r_{\rm CR}$'),
    ('pass_beta',  r'$+\ \beta \leq 1$ everywhere'),
    ('pass_wkb',   r'$+$ WKB in cavity'),
]

fig = plt.figure(figsize=(7, 2.2))
gs = gridspec.GridSpec(1, 4, figure=fig,
                    width_ratios=[1, 1, 1, 0.08],
                    wspace=0.05)
axes = [fig.add_subplot(gs[0, i]) for i in range(3)]
cax  = fig.add_subplot(gs[0, 3])   # asse dedicato alla colorbar

for ax, (col, title) in zip(axes, stages):
    fix_spines(ax)
    # ── build heatmap matrix ───────────────────────────────────────────
    frac_map = np.full((len(B00_GRID), len(SIGMA0_GRID)), np.nan)
    for i, B00 in enumerate(B00_GRID):
        for j, S0 in enumerate(SIGMA0_GRID):
            sub = df[(df['B00'] == B00) & (df['Sigma0'] == S0)]
            if len(sub) == 0:
                continue
            if col not in df.columns:
                frac_map[i, j] = 0.0
                continue
            frac_map[i, j] = sub[col].sum() / len(sub)

    # ── plot ───────────────────────────────────────────────────────────
    im = ax.pcolormesh(SIGMA0_GRID, B00_GRID, frac_map,
                        vmin=0, vmax=1, cmap='RdYlGn', shading='auto')

    # ── reference lines and cross ──────────────────────────────────────
    ax.axvline(SIGMA0_REF, color='white', ls='--', lw=1, alpha=0.85)
    ax.axhline(B00_REF,   color='white', ls='--', lw=1, alpha=0.85)
    ax.plot(SIGMA0_REF, B00_REF, 'x',
            color='white', ms=8, mew=1.5, zorder=3)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$\Sigma_0$  [g/cm²]')
    ax.set_title(title)

    # Nascondi le etichette y dei pannelli interni
    if col != 'pass_shear':
        ax.tick_params(labelleft=False)
        ax.sharey(axes[0])

axes[0].set_ylabel(r'$B_{00}$  [G]')

# Colorbar sull'asse dedicato → non tocca la larghezza dei pannelli
norm = Normalize(vmin=0, vmax=1)
cbar = fig.colorbar(ScalarMappable(norm=norm, cmap='RdYlGn'), cax=cax)
cbar.set_label('Fraction of spins')

plt.tight_layout()
for ext in ('pdf', 'png'):
    plt.savefig(f'aei_2/simple_heatmap_{hor}_{mm}.{ext}',
                bbox_inches='tight', dpi=150)
plt.show()

#### plot 2 - cfr H/R

In [ ]:
mm=2
cfgs_m = [c for c in CONFIGS if c['mm'] == mm]

fig = plt.figure(figsize=(7,2.8))
gs = gridspec.GridSpec(1, 3, figure=fig,
                    width_ratios=[1, 1, 0.05],
                    wspace=0.05)
axes = [fig.add_subplot(gs[0, i]) for i in range(2)]
cax  = fig.add_subplot(gs[0, 2])   # asse dedicato alla colorbar

# shared colour scale 0→1
vmin, vmax = 0.0, 1.0

for ax, cfg in zip(axes, cfgs_m):
    fix_spines(ax)
    hor = cfg['hor']
    key = (mm, hor)

    if key not in all_results:
        ax.text(0.5, 0.5, f'Dati mancanti\n(mm={mm}, hor={hor})',
                ha='center', va='center', transform=ax.transAxes)
        continue

    df = all_results[key]

    # ── build heatmap ──────────────────────────────────────────────────
    frac_map = np.full((len(B00_GRID), len(SIGMA0_GRID)), np.nan)
    for i, B00 in enumerate(B00_GRID):
        for j, S0 in enumerate(SIGMA0_GRID):
            sub = df[(df['B00'] == B00) & (df['Sigma0'] == S0)]
            if len(sub) == 0:
                continue
            frac_map[i, j] = sub['pass_wkb'].sum() / len(sub)

    n_valid_cells = int(np.sum(frac_map > 0))
    n_cells       = frac_map.size

    im = ax.pcolormesh(
        SIGMA0_GRID, B00_GRID, frac_map,
        vmin=vmin, vmax=vmax,
        cmap='RdYlGn', shading='auto',
    )

    # ── reference lines and cross ──────────────────────────────────────
    ax.axvline(SIGMA0_REF, color='white', ls='--', lw=1, alpha=0.9)
    ax.axhline(B00_REF,   color='white', ls='--', lw=1, alpha=0.9)
    ax.plot(SIGMA0_REF, B00_REF, 'x',
            color='white', ms=8, mew=1.5, zorder=4)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$\Sigma_0$  [g/cm²]')

    ax.text(0.97, 0.03,
        rf'$H/R = {hor}$',
        transform=ax.transAxes, va='bottom',
        horizontalalignment='right',
        color='black',
        bbox=dict(boxstyle='round', fc='white', alpha=0.5))

    # Nascondi le etichette y dei pannelli interni
    if hor == 0.001:
        ax.tick_params(labelleft=False)
        ax.sharey(axes[0])

axes[0].set_ylabel(r'$B_{00}$  [G]')

# Colorbar sull'asse dedicato → non tocca la larghezza dei pannelli
norm = Normalize(vmin=0, vmax=1)
cbar = fig.colorbar(ScalarMappable(norm=norm, cmap='RdYlGn'), cax=cax)
cbar.set_label('Fraction of spins')

plt.tight_layout()
for ext in ('pdf', 'png'):
    plt.savefig(f'aei_2/wkb_heatmap_comparison.{ext}', bbox_inches='tight', dpi=150)
plt.show()

## NT

In [ ]:
from aei_2.nt_disc import disk_model_NT

In [ ]:
CONF_nT = [
    dict(mm=1, color='#3b82f6', ls='--',  label=r'$m=1$'),
    #dict(mm=2,color='#ef4444', ls='.', label=r'$m=2$'),
]

MDOT_SCALE_NT = 10    # eta=0.1
# Parameter grids
N_MDOT  = 24
N_ALPHA = 18
MDOT_PLOT  = np.logspace(-5, 0, N_MDOT)   # physical mdot
MDOT_GRID = MDOT_PLOT * MDOT_SCALE_NT                 # mdot grid for NT disc (scaled up)
ALPHA_GRID = np.logspace(-3, 0, N_ALPHA)                    # alpha_visc

In [ ]:
def dQdr_nt(r_vec, a, mdot, alpha):
    """
    Compute dQ/dr = d/dr [Omega_phi * Sigma / B0²].
    The sign is independent of B00, Sigma0 (positive multiplicative constant).
    H/r (hor) does not enter Q at all.
    """
    res = make_disk(disk_model_NT, (r_vec, a, mdot, alpha))
    B0i = _make_interp(r_vec, res[0])
    Si  = _make_interp(r_vec, res[1])
    return compute_dQdr(r_vec, a, B0i, Si, M_BH)

# ═══════════════════════════════════════════════════════════════════════════════
# 2 ── MAIN SCAN
# ═══════════════════════════════════════════════════════════════════════════════

def cavity_nt(mm, verbose=False):
    records = []
    n_spin = len(A_GRID)
    n_par  = len(MDOT_GRID) * len(ALPHA_GRID)

    for i_a, a_val in enumerate(A_GRID):
        rISCO = float(r_isco(a_val))
        r_ILR, r_OLR, r_CR = get_resonances(a_val, mm)

        if verbose:
            print(f'  spin {i_a+1}/{n_spin}  a={a_val:.2f} '
                  f'| r_ILR={r_ILR:.1f}  r_OLR={r_OLR:.1f}  r_CR={r_CR:.1f}',
                  end='  ')

        # ── sanity check on resonance radii ───────────────────────────────
        if not (np.isfinite(r_OLR) and np.isfinite(r_ILR)):
            if verbose:
                print('SKIP (resonances not found)')
            # fill records with failed rows
            for mdot in MDOT_GRID:
                for alpha in ALPHA_GRID:
                    records.append(dict(a=a_val, mdot=mdot, alpha=alpha,
                                        r_ILR=np.nan, r_OLR=np.nan, r_CR=r_CR,
                                        dQdr_CR=np.nan,
                                        pass_shear=False, pass_beta=False,
                                        pass_wkb=False))
            continue

        if r_ILR <= rISCO * 1.001:
            if verbose:
                print('SKIP (ILR inside ISCO)')
            for mdot in MDOT_GRID:
                for alpha in ALPHA_GRID:
                    records.append(dict(a=a_val, mdot=mdot, alpha=alpha,
                                        r_ILR=r_ILR, r_OLR=r_OLR, r_CR=r_CR,
                                        dQdr_CR=np.nan,
                                        pass_shear=False, pass_beta=False,
                                        pass_wkb=False))
            continue

        # ─────────────────
        #r_olr_cap = min(r_OLR, rISCO * 500.0)
        #r_full    = np.geomspace(rISCO*1.001, r_olr_cap, N_R)
        r_ilr_cap = r_ILR*0.98
        r_cavity  = np.geomspace(rISCO*1.01, r_ilr_cap, N_R_CAV)

        # ── loop over (mdot, alpha) ──────────────────────────
        for mdot in MDOT_GRID:
            for alpha in ALPHA_GRID:
                """ no need for this cause always satisfied
                # ── Step A: beta ≤ 1 in [ISCO, OLR] ─────────────────────
                res_full = make_disk(disk_model_NT, (r_full, a_val, mdot, alpha))
                hr_full  = res_full[3]   # per-point H/r from model
                beta_full = compute_beta(res_full[0], res_full[1], res_full[2],
                                         r_full, hr_full, M_BH)
                pass_beta = bool(np.all(beta_full <= 1.0))
                beta_max  = float(np.max(beta_full))

                if not pass_beta:
                    records.append(dict(a=a_val, mdot=mdot, alpha=alpha,
                                        r_ILR=r_ILR, r_OLR=r_OLR, r_CR=r_CR,
                                        dQdr_CR=np.nan,
                                        pass_shear=False, pass_beta=False,
                                        pass_wkb=False, beta_max=beta_max))
                    continue

                # ── Step B: shear at CR ────────────────────
                r_cr_eps = np.linspace(r_CR*0.95, r_CR*1.05, 20)
                dq_full  = dQdr_nt(r_cr_eps, a_val, mdot, alpha)
                dqdr_at_CR = float(np.interp(r_CR, r_cr_eps, dq_full))
                pass_shear = bool(dqdr_at_CR > 0)

                if verbose:
                    shear_str = 'SHEAR OK' if pass_shear else 'shear FAIL'
                    print(shear_str)

                if not pass_shear:
                    for mdot in MDOT_GRID:
                        for alpha in ALPHA_GRID:
                            records.append(dict(a=a_val, mdot=mdot, alpha=alpha,
                                                r_ILR=r_ILR, r_OLR=r_OLR, r_CR=r_CR,
                                                dQdr_CR=dqdr_at_CR,
                                                pass_shear=False, pass_beta=True,
                                                pass_wkb=False))
                    continue

                """
                # ── Step C: WKB in cavity [ISCO, ILR] ────────────────────
                res_cav  = make_disk(disk_model_NT, (r_cavity, a_val, mdot, alpha))
                B0_c, Sg_c, cs_c, hr_c = res_cav[:4]
                info = res_cav[5]                

                k_arr = solve_k_aei(r_cavity, a_val, B0_c, Sg_c, cs_c,
                                    m=mm, M=M_BH)
                k_max_arr = 1 / np.maximum(hr_c, 1e-10)  # 1/hr(r)

                wkb_ok   = (np.isfinite(k_arr)
                            & (k_arr >= mm)
                            & (k_arr <= k_max_arr))
                pass_wkb = bool(np.all(wkb_ok))
                frac_ok  = float(wkb_ok.sum() / len(wkb_ok))

                k_valid = k_arr[np.isfinite(k_arr)]
                records.append(dict(
                    a=a_val, mdot=mdot, alpha=alpha,
                    r_ILR=r_ILR, r_OLR=r_OLR, r_CR=r_CR,
                    pass_shear=True, pass_beta=True, pass_wkb=pass_wkb,
                    frac_wkb_ok=frac_ok,
                    B00=info['B_ref'], Sigma0=info['Sigma_ref']
                ))

    return pd.DataFrame(records)


# ═══════════════════════════════════════════════════════════════════════════════
# 3 ── RUN ALL CONFIGS
# ═══════════════════════════════════════════════════════════════════════════════

def run_nt():
    all_results = {}
    for cfg in CONF_nT:
        mm = cfg['mm']
        print(f"\n{'='*65}")
        print(f"  Config: m={mm}")
        print(f"{'='*65}")
        df = cavity_nt(mm, verbose=False)
        all_results[mm] = df

        n      = len(df)
        n_sh   = df['pass_shear'].sum()
        n_be   = df.get('pass_beta',  pd.Series(dtype=bool)).sum() if 'pass_beta' in df else 0
        n_wkb  = df.get('pass_wkb',   pd.Series(dtype=bool)).sum() if 'pass_wkb'  in df else 0
        print(f'\n  Total combos : {n}')
        print(f'  Pass shear   : {n_sh:>6}  ({100*n_sh/n:.1f}%)')
        print(f'  + beta ≤ 1   : {n_be:>6}  ({100*n_be/n:.1f}%)')
        print(f'  + WKB cavity : {n_wkb:>6}  ({100*n_wkb/n:.1f}%)')

    return all_results

#### calc

In [ ]:
# ── step 1: full scan ─────────────────────────────────────────────────────
print('\n> Running cavity scan for all configs...')
all_nt = run_nt()

#### plot 1 - alpha mdot

In [ ]:
cfg = CONF_nT[0]
mm = cfg['mm']
df = all_nt[mm]

stages = [
    ('pass_wkb',   r'$+$ WKB in cavity'),
]

fig = plt.figure(figsize=(3.4, 3.1))
gs = gridspec.GridSpec(1, 2, figure=fig,
                    width_ratios=[1, 0.05],
                    wspace=0.05)
axes = [fig.add_subplot(gs[0, i]) for i in range(1)]
cax  = fig.add_subplot(gs[0, 1])   # asse dedicato alla colorbar

for ax, (col, title) in zip(axes, stages):
    fix_spines(ax)
    # ── build heatmap matrix ───────────────────────────────────────────
    frac_map = np.full((len(MDOT_GRID), len(ALPHA_GRID)), np.nan)
    for i, mdot in enumerate(MDOT_GRID):
        for j, alpha in enumerate(ALPHA_GRID):
            sub = df[(df['mdot'] == mdot) & (df['alpha'] == alpha)]
            if len(sub) == 0:
                continue
            if col not in df.columns:
                frac_map[i, j] = 0.0
                continue
            frac_map[i, j] = sub[col].sum() / len(sub)

    # ── plot ───────────────────────────────────────────────────────────
    X, Y = np.meshgrid(MDOT_PLOT, ALPHA_GRID)
    im = ax.pcolormesh(X, Y, frac_map.T,
                    vmin=0, vmax=1, cmap='RdYlGn', shading='auto')
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$\dot{m}$')

axes[0].set_ylabel(r'$\alpha$')

# Colorbar sull'asse dedicato → non tocca la larghezza dei pannelli
norm = Normalize(vmin=0, vmax=1)
cbar = fig.colorbar(ScalarMappable(norm=norm, cmap='RdYlGn'), cax=cax)
cbar.set_label('Fraction of spins')

plt.tight_layout()
for ext in ('pdf', 'png'):
    plt.savefig(f'aei_2/wkb_nt_mdot_alpha_{mm}.{ext}',
                bbox_inches='tight', dpi=150)
plt.show()

#### plot 2 - spin

In [ ]:
mm=1
hor = 0.001

fig, axes = plt.subplots(figsize=(3.4, 2.8))

fix_spines(axes)
df = all_nt[mm]
# ── build lineplot ──────────────────────────────────────────────────
frac_map = df.groupby('a')['pass_wkb'].mean().values
a_vals   = df.groupby('a')['pass_wkb'].mean().index.values
axes.plot(a_vals, frac_map, color='blue', lw=1, label="Shakura-Sunyaev disc")

key = (mm, hor)
df_2 = all_results[key]
frac_simp = np.full((len(A_GRID)), np.nan)
for i, A in enumerate(A_GRID):
    sub = df_2[(df_2['a'] == A)]
    if len(sub) == 0:
        continue
    frac_simp[i] = sub['pass_wkb'].sum() / len(sub)
axes.plot(A_GRID, frac_simp, color='red', lw=1, label="Simple disc")

axes.axhline(0, color='gray', ls='--', lw=1, alpha=0.7)
#axes.axhline(1, color='gray', ls='--', lw=1, alpha=0.7)

axes.set_xlabel(r'$a$')
axes.set_ylabel(r'Fraction of valid solutions')
axes.legend()

plt.tight_layout()
for ext in ('pdf', 'png'):
    plt.savefig(f'aei_2/sol_spin.{ext}', bbox_inches='tight', dpi=150)
plt.show()

#### plot 3 - B00, Sigma0

In [ ]:
cmap='RdBu_r'
alpha_pt=0.55
s_pt=14
figsize=(3.4, 2.9)
outdir='aei_2'

df_plot = all_nt[mm]

a_vals  = df_plot['a'].values
S_vals  = df_plot['Sigma0'].values
B_vals  = df_plot['B00'].values

# colour mapped on spin: symmetric range [-1, 1]
norm = Normalize(vmin=-1.0, vmax=1.0)
cm   = plt.get_cmap(cmap)

# ── figure ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=figsize)

sc = ax.scatter(S_vals, B_vals,
                c=a_vals, cmap=cm, norm=norm,
                s=s_pt, alpha=alpha_pt,
                linewidths=0, rasterized=True, zorder=3)

cb = plt.colorbar(ScalarMappable(norm=norm, cmap=cm), ax=ax, pad=0.02)
cb.set_label(r'$a$')

# ── reference lines / cross ───────────────────────────────────────────
ax.axvline(SIGMA0_REF, color='black', ls='--', lw=1, alpha=0.8)
ax.axhline(B00_REF, color='black', ls='--', lw=1, alpha=0.8)
ax.plot(SIGMA0_REF, B00_REF, 'kx', ms=8, mew=1.5, zorder=4)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'$\Sigma_0$  [g cm$^{-2}$]')
ax.set_ylabel(r'$B_{\rm 00}$  [G]')

stage_labels = {
    'pass_shear': r'$dQ/dr > 0$',
    'pass_beta':  r'$dQ/dr > 0$ + $\beta \leq 1$',
    'pass_wkb':   r'all checks (+ WKB cavity)',
}

plt.tight_layout()

# ── save ──────────────────────────────────────────────────────────────
for ext in ('pdf', 'png'):
    fpath = f'{outdir}/scatter_bisco_sigma_NT.{ext}'
    plt.savefig(fpath, bbox_inches='tight', dpi=150)

plt.show()

#### radial profiles

In [ ]:
ZONE_COLOR = {'A': '#3b82f6', 'B': '#f97316', 'C': '#22c55e'}
from matplotlib.patches import Patch

def plot_nt_profiles(
    a      = 0.9,
    mdot   = 0.02,
    alpha  = 0.1,
    M      = M_BH,
    n_r    = 600,
    r_min_fac = 1.001,
    r_max_fac = 250.0,
    show_Q_markers = False,   # mostra r(Q_max) e r(Q_min) — off di default
    save   = True,
    outdir = 'aei_2/',
    outfile = None,           # None → nome automatico
    figsize = (7, 6),
):
    # ── griglia radiale ───────────────────────────────────────────────────
    rISCO = float(r_isco(a))
    r_vec = np.geomspace(rISCO * r_min_fac, rISCO * r_max_fac, n_r)

    # ── profili fisici ────────────────────────────────────────────────────
    B0_arr, Sig_arr, cs_arr, hr_arr, zone_arr, info_d = disk_model_NT(
        r_vec, a, mdot, alpha_visc=alpha, hr=None, M=M
    )
    r_AB = info_d['r_AB']
    r_BC = info_d['r_BC']

    # ── risonanze ─────────────────────────────────────────────────────────
    r_ILR = r_ilr(a, M=M)
    r_OLR = r_olr(a, M=M)

    # ── marcatori verticali ───────────────────────────────────────────────
    markers = [(rISCO, 'black', ':', r'$r_{\rm ISCO}$', 0.5)]
    if np.isfinite(r_ILR):
        markers.append((r_ILR, '#06b6d4', '--', r'$r_{\rm ILR}$', 1.8))
    if np.isfinite(r_OLR):
        markers.append((r_OLR, '#d946ef', '-.', r'$r_{\rm OLR}$', 1.8))
    if r_AB > rISCO * 1.01:
        markers.append((r_AB, '#f97316', ':', r'$r_{AB}$', 1.5))
    if r_BC > r_AB * 1.01:
        markers.append((r_BC, '#22c55e', ':', r'$r_{BC}$', 1.5))

    # ── panels: (data, ylabel, log_y, hlines) ────────────────────────────
    panels = [
        (B0_arr,   r'$B_0$  [G]',              True,  []),
        (Sig_arr,  r'$\Sigma$  [g cm$^{-2}$]', True,  []),
        (hr_arr,   r'$H/R$',                   True,
            [(0.1, 'steelblue', ':', r'$H/r=0.1$'),
             (0.3, 'crimson',   ':', r'$H/r=0.3$')]),
    ]

    # ── figura ────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(
        3, 1, figsize=figsize, sharex=True,
        gridspec_kw=dict(hspace=0.06),
    )

    def _shade(ax):
        for zname, zlo, zhi in [('A', rISCO, r_AB),
                                 ('B', r_AB,  r_BC),
                                 ('C', r_BC,  r_vec[-1])]:
            if zhi > zlo:
                ax.axvspan(zlo, zhi, alpha=0.2,
                           color=ZONE_COLOR.get(zname, 'gray'))

    def _vlines(ax):
        for rv, col, ls, lbl, lw in markers:
            ax.axvline(rv, color=col, ls=ls, lw=lw, alpha=0.85)

    for k, (ax, (data, ylabel, log_y, hlines)) in enumerate(zip(axes, panels)):
        _shade(ax)
        _vlines(ax)
        fix_spines(ax)

        for zname, col in ZONE_COLOR.items():
            mask = zone_arr == zname
            ok   = mask & (data > 0) if log_y else mask
            if ok.sum() >= 2:
                ax.plot(r_vec[ok], data[ok], color=col, lw=2.0, zorder=3)

        for yval, hcol, hls, hlbl in hlines:
            ax.axhline(yval, color=hcol, ls=hls, lw=1.4, alpha=0.85,
                       label=hlbl)

        ax.set_xscale('log')
        if log_y:
            ax.set_yscale('log')
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.15, which='both')
        #ax.set_xlim(r_vec[0], r_vec[-1])
        if k < 2:
            ax.tick_params(axis='x', labelbottom=False)

    axes[-1].set_xlabel(r'$R$  [$r_g$]')

    # ── legenda (pannello superiore) ──────────────────────────────────────
    legend_els = [
        Line2D([0], [0], color=col, ls=ls, lw=lw, label=lbl)
        for rv, col, ls, lbl, lw in markers
    ] + [
        Patch(facecolor=col, alpha=0.4, label=f'Zone {z}')
        for z, col in ZONE_COLOR.items()
    ]
    axes[0].legend(handles=legend_els, ncol=2,
                   loc='upper right', framealpha=0.88,
                   borderpad=0.5, labelspacing=0.25, handlelength=1.6)

    plt.tight_layout()

    # ── salvataggio opzionale ─────────────────────────────────────────────
    if save:
        import os
        os.makedirs(outdir, exist_ok=True)
        base = outfile or f'nt_profiles_a{a:.2f}_mdot{mdot:.3g}_alpha{alpha}'
        base = os.path.join(outdir, base)
        for ext in ('pdf', 'png'):
            fig.savefig(f'{base}.{ext}', bbox_inches='tight', dpi=150)
            print(f'  → {base}.{ext}')

    # ── summary numerico ──────────────────────────────────────────────────
    out_info = dict(rISCO=rISCO, r_AB=r_AB, r_BC=r_BC,
                    r_ILR=r_ILR, r_OLR=r_OLR)
    return fig, out_info

In [ ]:
# uso base
fig, info = plot_nt_profiles(a=0.9, mdot=0.02, alpha=0.1)